In [ ]:
import itertools
%autoreload 2

In [1]:
%reload_ext autoreload
import os, sys, random
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

import matplotlib.pyplot as plt
from matplotlib import gridspec, rcParams
from fish import Gafftopsail
sys.path.append(r'/Users/zichenhe/miniforge3/envs/naumann_lab/2ptank/')#(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')
import utils, barcode
path = ('C://Data//Imaging//260425_overlap/fish6_2/')
rcParams['font.size'] = 10

In [2]:
fish = Gafftopsail(path, filelist = ['stimulus', 'imaging'], sequence = 5)
fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]

low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed


__EACH STIM__

In [3]:
#axis 0: trial; axis 1: neuron; axis 2: frame
#from stationary start to duration + 20
f_pertrial_dict = barcode.get_pertrial_f(fish)

getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s


In [4]:
#avg activity
f_avg_dict = {s: np.nanmean(f_pertrial_dict[s], axis = 0) for s in f_pertrial_dict.keys()}

__get visually responsive cells__

In [5]:
vis_neurons = barcode.select_visbarcode(fish, f_pertrial_dict, baseline_s = 5, response_s =20, perc_trial_threshold = 0.8)

selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1


In [ ]:
def get_ols_multiplex(vis_neurons, f_avg_dict):
    grating_stims = ['forward', 'left', 'right', 'backward']
    dot_stims = ['dot_l', 'dot_r']
    ols_df = {'neuron_index': vis_neurons}
    for row, g_stim in enumerate(grating_stims):
        for col, d_stim in enumerate(dot_stims):
            o_stim = str([d_stim, g_stim])
            g_response = f_avg_dict[g_stim]
            d_response = f_avg_dict[d_stim]
            o_response = f_avg_dict[o_stim]
            glist = []
            dlist = []
            rlist = []
            for n in vis_neurons:
                y = o_response[n]
                x1 = g_response[n]
                x2 = d_response[n]
                X = np.column_stack([x1, x2])
                X = sm.add_constant(X)
                result = sm.OLS(y, X).fit()
                a = result.params[1]   # coefficient for x1
                b = result.params[2]   # coefficient for x2
                r2 = result.rsquared
                glist.append(a)
                dlist.append(b)
                rlist.append(r2)
            ols_df[f'{o_stim}_grating_coe'] = glist
            ols_df[f'{o_stim}_dot_coe'] = dlist
            ols_df[f'{o_stim}_response_coe'] = rlist
    ols_df = pd.DataFrame.from_dict(ols_df)
    return ols_df
ols_df = get_ols_multiplex(vis_neurons, f_avg_dict)
ols_df.to_csv(fish.path + '//analyze_results//ols.csv', index = False)

In [28]:
def get_ols_multiplex_bytrial(vis_neurons, f_avg_dict, f_pertrial_dict):
    grating_stims = ['forward', 'left', 'right', 'backward']
    dot_stims = ['dot_l', 'dot_r']
    ols_df = {}
    for row, g_stim in enumerate(grating_stims):
        for col, d_stim in enumerate(dot_stims):
            o_stim = str([d_stim, g_stim])
            g_response = f_avg_dict[g_stim]
            d_response = f_avg_dict[d_stim]
            o_response = f_pertrial_dict[o_stim]
            glist = []
            dlist = []
            rlist = []
            nlist = []
            triallist = []
            for n in vis_neurons:
                o_response_n = o_response[:, n]
                x1 = g_response[n]
                x2 = d_response[n]
                X = np.column_stack([x1, x2])
                X = sm.add_constant(X)
                for trial in range(5):
                    if trial >= f_pertrial_dict[o_stim].shape[0]:
                        a = np.nan
                        b = np.nan
                        r2 = np.nan
                    else:
                        y = o_response_n[trial]
                        result = sm.OLS(y, X).fit()
                        a = result.params[1]   # coefficient for x1
                        b = result.params[2]   # coefficient for x2
                        r2 = result.rsquared
                    glist.append(a)
                    dlist.append(b)
                    rlist.append(r2)
                    nlist.append(n)
                    triallist.append(trial)
            ols_df[f'{o_stim}_grating_coe'] = glist
            ols_df[f'{o_stim}_dot_coe'] = dlist
            ols_df[f'{o_stim}_response_coe'] = rlist
            ols_df[f'{o_stim}_neuron_index'] = nlist
            ols_df[f'{o_stim}_trial_index'] = triallist
    [print(len(ols_df[k]), k) for k in ols_df.keys()]
    ols_df = pd.DataFrame.from_dict(ols_df)
    return ols_df
#ols_df_bytrial = get_ols_multiplex_bytrial(vis_neurons, f_avg_dict, f_pertrial_dict)

In [29]:
fig, ax = plt.subplots(len(grating_stims), len(dot_stims), figsize = (10, 20), dpi = 200)
vis_neurons = ols_df.neuron_index
for row, g_stim in enumerate(grating_stims):
    for col, d_stim in enumerate(dot_stims):
        ax_stim = ax[row,col]
        dlist = np.abs(ols_df.loc[:, f'{o_stim}_dot_coe'])
        glist = np.abs(ols_df.loc[:, f'{o_stim}_grating_coe'])
        dlist = np.divide(np.subtract(dlist, glist), np.add(dlist, glist))#np.subtract(np.abs(glist), np.abs(dlist))
        #plot
        vmax = max(np.abs(np.nanpercentile(dlist, 5)), np.abs(np.nanpercentile(dlist, 95)))
        vmin = -vmax
        ax_stim.imshow(fish.img_dict[3], cmap = 'gray', origin = 'lower')
        ax_stim.scatter(fish.pos_all.loc[vis_neurons, 'xpos'], fish.pos_all.loc[vis_neurons, 'ypos'], c = dlist, alpha =1, cmap = 'bwr', vmin = vmin, vmax = vmax, s = 1)
plt.show()
plt.close()

NameError: name 'grating_stims' is not defined

In [30]:
#combine across fishes
paths =[
 r"C:\Data\Imaging\260415_overlap\fish3",r"C:\Data\Imaging\260415_overlap\fish3_2",
r"C:\Data\Imaging\260415_overlap\fish4", r"C:\Data\Imaging\260415_overlap\fish4_2",
     r"C:\Data\Imaging\260415_overlap\fish6", r"C:\Data\Imaging\260415_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish1", r"C:\Data\Imaging\260425_overlap\fish1_2",
r"C:\Data\Imaging\260425_overlap\fish2",r"C:\Data\Imaging\260425_overlap\fish2_2",
r"C:\Data\Imaging\260425_overlap\fish3",r"C:\Data\Imaging\260425_overlap\fish3_2",
r"C:\Data\Imaging\260425_overlap\fish5",r"C:\Data\Imaging\260425_overlap\fish5_2",
 r"C:\Data\Imaging\260425_overlap\fish6", r"C:\Data\Imaging\260425_overlap\fish6_2",
r"C:\Data\Imaging\260425_overlap\fish7",r"C:\Data\Imaging\260425_overlap\fish7_2",
r"C:\Data\Imaging\260425_overlap\fish8",r"C:\Data\Imaging\260425_overlap\fish8_2",
r"C:\Data\Imaging\260425_overlap\fish9", r"C:\Data\Imaging\260425_overlap\fish9_2",
 r"C:\Data\Imaging\260425_overlap\fish10", r"C:\Data\Imaging\260425_overlap\fish10_2",
r"C:\Data\Imaging\260425_overlap\fish11", r"C:\Data\Imaging\260425_overlap\fish11_2",]
for path in paths:
    fish = Gafftopsail(path + '//', filelist=['stimulus', 'imaging'], sequence=5)
    fish.stimulus_df.stim_name = [str(s) for s in fish.stimulus_df.stim_name]

    #axis 0: trial; axis 1: neuron; axis 2: frame
    #from stationary start to duration + 20
    f_pertrial_dict = barcode.get_pertrial_f(fish)
    #avg activity
    f_avg_dict = {s: np.nanmean(f_pertrial_dict[s], axis=0) for s in f_pertrial_dict.keys()}
    vis_neurons = barcode.select_visbarcode(fish, f_pertrial_dict, baseline_s=5, response_s=20, perc_trial_threshold=0.8)


    #ols_df = get_ols_multiplex(vis_neurons, f_avg_dict)
    # ols_df.to_csv(fish.path + '//analyze_results//ols.csv', index=False)
    ols_df = get_ols_multiplex_bytrial(vis_neurons, f_avg_dict, f_pertrial_dict)
    ols_df.to_csv(fish.path + '//analyze_results//ols_bytrial.csv', index=False)

low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
low passs assuming framerate ~2
smoothed
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
getting pertrial f stim duration + 20s
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= baseline mean * 1
selection criteria: max >= bas